<h1>🔀 Biofilter — Report: <code>annotation_master_pathway</code></h1>

Everything the bundle knows about a list of pathways: canonical id and description, which source curated it, and what it is linked to.

### 1. Open a bundle

In [ ]:
from biofilter import Biofilter

# A bundle is a directory — the one holding manifest.json.
# Leave as None to use `[database] bundle` from .biofilter.toml.
BUNDLE = None
REPORT = "annotation_master_pathway"

bf = Biofilter(bundle=BUNDLE, debug_mode=False) if BUNDLE else Biofilter(debug_mode=False)
bf

### 2. What the report offers

In [ ]:
print("columns:")
for column in bf.report.available_columns(REPORT):
    print(" ", column)

print("\nexample input:")
print(bf.report.example_input(REPORT))

In [ ]:
print(bf.report.explain(REPORT))

### 3. Run it

Pathways resolve by id — Reactome (`R-HSA-…`) or KEGG (`hsa…`).

In [ ]:
pathways = [
    "R-HSA-109581",   # Apoptosis, Reactome
    "hsa04210",       # Apoptosis, KEGG
    "NOT_A_PATHWAY",  # kept, with status='not_found'
]

result = bf.report.run(REPORT, input_data=pathways)
df = result.to_pandas()
df[["input_value", "pathway_id", "pathway_description",
    "pathway_source_system", "status"]]

### 4. The same biology, curated twice

Reactome and KEGG describe overlapping biology with different granularity
and different ids, and the bundle carries both. Two rows can be the same
pathway under two curations — **nothing in this report merges them**, and
`pathway_source_system` is how you tell which is which.

In [ ]:
df[["pathway_id", "pathway_source_system", "pathway_data_source",
    "total_entity_relationships"]]

### 5. What makes a pathway useful

A pathway with a large `Genes` count is one the bundle can expand into a
gene set. One with none is present as a label only.

In [ ]:
for _, row in df[df["status"] == "ok"].iterrows():
    print(f"{row['pathway_id']}  {row['pathway_description']}")
    for entry in row["entity_relationships_by_group"]:
        print(f"    {entry['group_name']:<12} {entry['count']:>6}")
    print()

### 6. Every pathway in the bundle

In [ ]:
import time

started = time.perf_counter()
everything = bf.report.run(REPORT, input_data="__ALL__")
catalog = everything.to_pandas()

print(f"{everything.num_rows:,} pathways in {time.perf_counter() - started:.1f}s")
catalog["pathway_source_system"].value_counts()

### 7. Export

CSV by default, with a `.provenance.json` beside it naming the bundle the ids came from.

In [ ]:
for path in result.write("annotation_master_pathway.csv"):
    print(path)

### 8. The same thing on the command line

```bash
biofilter report run --report-name annotation_master_pathway \\
    --input ... \\
    --output out.csv
```

### 9. Quick QA

In [ ]:
expected = list(bf.report.available_columns(REPORT))
missing = [c for c in expected if c not in df.columns]

print("missing columns:", missing or "none")
print("unresolved inputs:", int((df["status"] == "not_found").sum()))
print("bundle:", result.provenance["bundle_id"])
display(df.dtypes.to_frame("dtype"))